Import libraries

In [2]:

import torch
import librosa
import numpy as np
import sounddevice
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path
import torch.nn as nn
from pythonosc import udp_client

Load trained autoencoders

In [4]:
#autoencoder class definition
class WindowAutoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=2, hidden_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model_rms = WindowAutoencoder()
model_rms.load_state_dict(torch.load("RMS/rms_autoencoder.pt"))
model_rms.eval()

model_centroid = WindowAutoencoder()
model_centroid.load_state_dict(torch.load("Centroid/centroid_autoencoder.pt"))
model_centroid.eval()

model_onset = WindowAutoencoder()
model_onset.load_state_dict(torch.load("ON-set/onset_autoencoder.pt"))
model_onset.eval()

WindowAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=20, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=2, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=20, bias=True)
  )
)

Feature extraction utilities 

In [6]:
#extract windows
def extract_windows(feature_array, window_size=20, step=1):
    windows = []
    for i in range(0, len(feature_array) - window_size, step):
        windows.append(feature_array[i:i + window_size])
    return np.array(windows)

def extract_all_features(clip_path):
    y, sr = librosa.load(clip_path, sr=44100, mono=True)
    rms = librosa.feature.rms(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    return extract_windows(rms), extract_windows(centroid), extract_windows(onset_env)

def encode_clip(rms_w, centroid_w, onset_w):
    def avg_latent(windows, model):
        x = torch.tensor(windows, dtype=torch.float32)
        with torch.no_grad():
            _, z = model(x)
        return z.numpy().mean(axis=0)
    return {
        "RMS": avg_latent(rms_w, model_rms),
        "Centroid": avg_latent(centroid_w, model_centroid),
        "Onset": avg_latent(onset_w, model_onset)
    }

Combine features in one encoded representation 


In [7]:
def encode_moment(rms_window, centroid_window, onset_window):
    with torch.no_grad():
        _, z_rms = model_rms(torch.tensor(rms_window, dtype=torch.float32).unsqueeze(0))
        _, z_centroid = model_centroid(torch.tensor(centroid_window, dtype=torch.float32).unsqueeze(0))
        _, z_onset = model_onset(torch.tensor(onset_window, dtype=torch.float32).unsqueeze(0))
        
        #audio_tensor = torch.from_numpy(audio_chunk).float().unsqueeze(0).unsqueeze(0)
        #z_rave = rave_model.encode(audio_tensor)
    
    return {
        "rms": z_rms.numpy(),
        "centroid": z_centroid.numpy(),
        "onset": z_onset.numpy(),
        #"rave": z_rave.numpy()
    }

Process an entire audio file 

In [8]:
#encode
test_path = "daria-recordings/chitara-pt-cod.wav"
y, sr = librosa.load(test_path, sr=44100, mono=True)

rms = librosa.feature.rms(y=y)[0]
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
onset_env = librosa.onset.onset_strength(y=y, sr=sr)

rms_windows = extract_windows(rms)
centroid_windows = extract_windows(centroid)
onset_windows = extract_windows(onset_env)

audio_chunk = y[:2048]

result = encode_moment(rms_windows[0], centroid_windows[0], onset_windows[0])

for k, v in result.items():
    print(k, v.shape, v)

rms (1, 2) [[-0.610545  -0.2775543]]
centroid (1, 2) [[-4566.6616  4965.864 ]]
onset (1, 2) [[2.0401077 5.042877 ]]


In [9]:
def encode_full_clip(clip_path, window_size=20):
    y, sr = librosa.load(clip_path, sr=44100, mono=True)
    
    rms = librosa.feature.rms(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    
    rms_windows = extract_windows(rms, window_size)
    centroid_windows = extract_windows(centroid, window_size)
    onset_windows = extract_windows(onset_env, window_size)
    
    n_steps = min(len(rms_windows), len(centroid_windows), len(onset_windows))
    
    results = []
    for i in range(n_steps):
        result = encode_moment(rms_windows[i], centroid_windows[i], onset_windows[i])
        results.append(result)
    
    return results

In [10]:
all_results = encode_full_clip("daria-recordings/chitara-pt-cod.wav")
print(len(all_results))
print(all_results[0])
print(all_results[-1])

1986
{'rms': array([[-0.610545 , -0.2775543]], dtype=float32), 'centroid': array([[-4566.6616,  4965.864 ]], dtype=float32), 'onset': array([[2.0401077, 5.042877 ]], dtype=float32)}
{'rms': array([[-0.6210908, -0.2824462]], dtype=float32), 'centroid': array([[-6061.127,  6599.421]], dtype=float32), 'onset': array([[-2.401049 , -3.4582982]], dtype=float32)}


Live pipeline: microphone → features → autoencoders → OSC → Processing

In [22]:
import sounddevice as sd
from collections import deque
import numpy as np

sd.default.device = [0, None]

sr = 44100
block_size = 512
window_size = 20
audio_history_size = 4096

rms_buffer = deque(maxlen=window_size)
centroid_buffer = deque(maxlen=window_size)
onset_buffer = deque(maxlen=window_size)
audio_history = deque(maxlen=audio_history_size)

def callback(indata, frames, time, status):
    y = indata[:, 0]
    audio_history.extend(y)
    
    rms_val = librosa.feature.rms(y=y, frame_length=block_size, hop_length=block_size)[0][0]
    centroid_val = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=block_size, hop_length=block_size)[0][0]
    
    if len(audio_history) == audio_history_size:
        y_history = np.array(audio_history)
        onset_env = librosa.onset.onset_strength(y=y_history, sr=sr, hop_length=block_size)
        onset_val = onset_env[-1] if len(onset_env) > 0 else 0.0
    else:
        onset_val = 0.0
    
    rms_buffer.append(rms_val)
    centroid_buffer.append(centroid_val)
    onset_buffer.append(onset_val)
    
    if len(rms_buffer) == window_size:
        result = encode_moment(np.array(rms_buffer), np.array(centroid_buffer), np.array(onset_buffer))
        
        osc_client.send_message("/rms", result["rms"].flatten().tolist())
        osc_client.send_message("/centroid", result["centroid"].flatten().tolist())
        osc_client.send_message("/onset", result["onset"].flatten().tolist())
        
        print(result)

with sd.InputStream(samplerate=sr, blocksize=block_size, channels=1, device=0, callback=callback):
    print("Listening...")
    sd.sleep(60000)

Listening...
{'rms': array([[-0.30205378, -0.22779775]], dtype=float32), 'centroid': array([[-23262.242,  21957.959]], dtype=float32), 'onset': array([[4.8502674, 2.892574 ]], dtype=float32)}
{'rms': array([[-0.30244413, -0.22909728]], dtype=float32), 'centroid': array([[-24013.713,  23957.342]], dtype=float32), 'onset': array([[4.1234736, 2.3143935]], dtype=float32)}
{'rms': array([[-0.30247208, -0.22895917]], dtype=float32), 'centroid': array([[-25530.354,  23517.89 ]], dtype=float32), 'onset': array([[2.6314914, 2.302838 ]], dtype=float32)}
{'rms': array([[-0.30235723, -0.22867194]], dtype=float32), 'centroid': array([[-26579.771,  21449.672]], dtype=float32), 'onset': array([[0.9225202, 2.422783 ]], dtype=float32)}
{'rms': array([[-0.30232945, -0.22853181]], dtype=float32), 'centroid': array([[-27914.48 ,  22184.414]], dtype=float32), 'onset': array([[0.40627545, 1.4218819 ]], dtype=float32)}
{'rms': array([[-0.3022644 , -0.22831348]], dtype=float32), 'centroid': array([[-29049.273